# Fake News Detection: TF-IDF baselines vs Transformers (DistilBERT, BERT, RoBERTa, DeBERTa)

**Goal:** classify a news article as **REAL** or **FAKE**, and measure honestly how well the model works (a) on unseen WELFake articles and (b) on a different dataset (LIAR).

**Datasets**
- **WELFake (required):** ~72,000 news articles. Columns: `title`, `text`, `label`. Used for training and the main test.
- **LIAR (optional):** ~12,800 short political statements. Used only as a second, unseen test. *You can run this notebook with WELFake alone; the LIAR cells skip themselves if the files are missing.*

**Why not Fake.csv + True.csv (ISOT)?** Real articles there start with "CITY (Reuters) -", so models learn that shortcut and score about 100%.

**What this notebook gives you**
1. Dataset screenshot cell (file name, dataset name, column names, label counts).
2. Six models, one protocol: TF-IDF + LogReg, TF-IDF + SVM, DistilBERT, BERT, RoBERTa, DeBERTa.
3. For **every model**: classification report, confusion matrix, ROC curve (printed and saved as PNG).
4. Comparison bar charts, grid of all confusion matrices and ROC curves, cross-dataset chart.
5. A verification cell that proves the numbers are internally consistent.
6. Everything saved to Google Drive.

Run: Runtime -> Change runtime type -> **T4 GPU**, then **Runtime -> Run all** (about 1 to 1.5 hours in total). Keep the tab open.

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn

In [ ]:
import os, re, glob, random, warnings, gc, pickle, shutil
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
from datasets import Dataset
from scipy.special import softmax
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, roc_curve, auc, ConfusionMatrixDisplay)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
                          Trainer, DataCollatorWithPadding)
warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE (enable T4 GPU)")

## Step 1: Connect Google Drive and find the data

Put your data in Drive **once**: `MyDrive/FakeNews_Project/data/welfake/WELFake_Dataset.csv` (and, optionally, `data/liar/train.tsv, valid.tsv, test.tsv`).
If you already uploaded the folders to the Colab Files panel as `data/welfake` and `data/liar`, that works too.

Download links:
- WELFake: https://www.kaggle.com/datasets/studymart/welfake-dataset-for-fake-news
- LIAR (optional): https://www.kaggle.com/datasets/doanquanvietnamca/liar-dataset

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/FakeNews_Project"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Google Drive not mounted:", e)

LOCAL_DATA = "data"
if not os.path.exists(LOCAL_DATA) and os.path.exists(f"{DRIVE_ROOT}/data"):
    shutil.copytree(f"{DRIVE_ROOT}/data", LOCAL_DATA)
    print("Copied data from Drive to this session.")

# where results will be saved
SAVE_DIR = f"{DRIVE_ROOT}/results" if os.path.isdir("/content/drive/MyDrive") else "results"
os.makedirs(SAVE_DIR, exist_ok=True); os.makedirs("figs", exist_ok=True)
print("Results will be saved to:", SAVE_DIR)

print("\nFiles found:")
for p in sorted(glob.glob(f"{LOCAL_DATA}/**/*", recursive=True)):
    if os.path.isfile(p):
        print(f"  {p}   {os.path.getsize(p)/1e6:.1f} MB")

wel_files = [p for p in glob.glob(f"{LOCAL_DATA}/**/*.csv", recursive=True) if "wel" in os.path.basename(p).lower()]
assert wel_files, "WELFake_Dataset.csv not found. Put it in data/welfake/ (Files panel) or in Drive: MyDrive/FakeNews_Project/data/welfake/"
wel_path = max(wel_files, key=os.path.getsize)
size_mb = os.path.getsize(wel_path) / 1e6
print(f"\nUsing WELFake file: {wel_path} ({size_mb:.1f} MB)")
if size_mb < 200:
    print("WARNING: the file should be about 245 MB. It may be an incomplete upload. Delete it and upload again.")

## Step 2: Look at the data (take your screenshot here)

Prints **dataset name, file name, shape, column names, data types, label counts, text lengths** and the first rows.

In [ ]:
wel = pd.read_csv(wel_path)
print("DATASET NAME : WELFake (Word Embedding over Linguistic Features for Fake news detection)")
print("FILE NAME    :", os.path.basename(wel_path))
print("SHAPE        :", wel.shape, "(rows, columns)")
print("COLUMN NAMES :", list(wel.columns))
print("\nColumn meaning:")
print("  Unnamed: 0 -> row number (not used)")
print("  title      -> headline of the article (input)")
print("  text       -> full body of the article (input)")
print("  label      -> class, 0 or 1 (target we predict)")
print("\nData types:\n", wel.dtypes)
print("\nLabel counts:\n", wel["label"].value_counts())
print("\nMissing values:\n", wel.isna().sum())
lens = wel["text"].fillna("").str.split().str.len()
print("\nWords per article: mean %.0f, median %.0f, max %d" % (lens.mean(), lens.median(), lens.max()))
display(wel.head())

## Step 3: Check which label is fake

The WELFake page says 0 = fake, 1 = real, but many users report the file is the **opposite**. Read the samples below and set `LABEL_1_MEANS` in the next cell to `"fake"` or `"real"`.

In [ ]:
for lab in [0, 1]:
    print(f"\n===== label = {lab} =====")
    for t in wel[wel.label == lab]["title"].dropna().sample(8, random_state=1):
        print("-", t[:120])

In [ ]:
LABEL_1_MEANS = "fake"   # <- change to "real" if the samples above show that label 1 is real news
# Our convention below: FAKE = 0, REAL = 1

## Step 4: Clean, remove shortcut patterns, remove duplicates, split

In [ ]:
df = wel.copy()
df["title"] = df["title"].fillna("")
df["text"] = df["text"].fillna("")
df["content"] = (df["title"] + ". " + df["text"]).str.strip()
df["y"] = df["label"] if LABEL_1_MEANS == "real" else 1 - df["label"]   # FAKE=0, REAL=1

def clean(t):
    t = str(t)
    t = re.sub(r"^.{0,60}\((reuters|ap|afp)\)\s*[-–—]?\s*", "", t, flags=re.I)  # agency tags (leak)
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    return re.sub(r"\s+", " ", t).strip()

df["content"] = df["content"].map(clean)
before = len(df)
df = df[df["content"].str.split().str.len() >= 10]
df = df.drop_duplicates(subset="content")
print(f"rows: {before} -> {len(df)}  (removed very short rows and exact duplicates)")
print(df["y"].value_counts(normalize=True).rename({0: "FAKE", 1: "REAL"}))

train_full, test_full = train_test_split(df[["content", "y"]], test_size=0.2, stratify=df["y"], random_state=SEED)

TRAIN_N, TEST_N = 20000, 5000      # raise for slightly better results (needs more time)
train_df = train_full.sample(min(TRAIN_N, len(train_full)), random_state=SEED).rename(columns={"content": "text", "y": "label"}).reset_index(drop=True)
test_df  = test_full.sample(min(TEST_N, len(test_full)),  random_state=SEED).rename(columns={"content": "text", "y": "label"}).reset_index(drop=True)
print("train:", train_df.shape, " test:", test_df.shape)
print("train/test exact overlap:", len(set(train_df["text"]) & set(test_df["text"])))

## Step 5: Evaluation helper

For every model this prints the classification report and shows + saves the **confusion matrix and ROC curve** (PNG files go to the `figs` folder).

In [ ]:
results, roc_store, preds = {}, {}, {}            # in-domain (WELFake test)
ext_results, ext_roc_store, ext_preds = {}, {}, {}  # cross-dataset (LIAR test)
liar_in_results = {}                               # optional: trained and tested on LIAR

KINDS = {"in": (results, roc_store, preds, ""),
         "ext": (ext_results, ext_roc_store, ext_preds, " (LIAR)"),
         "liar_in": (liar_in_results, {}, {}, " (trained on LIAR)")}

def evaluate(name, y_true, y_pred, y_prob, kind="in", plot=True):
    store, rstore, pstore, suffix = KINDS[kind]
    y_true, y_pred, y_prob = np.asarray(y_true), np.asarray(y_pred), np.asarray(y_prob)
    fpr, tpr, _ = roc_curve(y_true, y_prob); a = auc(fpr, tpr)
    store[name] = dict(Accuracy=accuracy_score(y_true, y_pred),
                       Precision=precision_score(y_true, y_pred, zero_division=0),
                       Recall=recall_score(y_true, y_pred, zero_division=0),
                       F1=f1_score(y_true, y_pred, zero_division=0),
                       MacroF1=f1_score(y_true, y_pred, average="macro", zero_division=0), AUC=a)
    rstore[name] = (fpr, tpr, a); pstore[name] = (y_true, y_pred, y_prob)
    print(f"\n===== {name}{suffix} =====   N = {len(y_true)}")
    print(classification_report(y_true, y_pred, target_names=["FAKE", "REAL"], digits=4, zero_division=0))
    if plot:
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=["FAKE", "REAL"]).plot(ax=ax[0], cmap="Blues", colorbar=False)
        ax[0].set_title(f"Confusion matrix: {name}{suffix}")
        ax[1].plot(fpr, tpr, label=f"AUC = {a:.4f}"); ax[1].plot([0, 1], [0, 1], "k--")
        ax[1].set_xlabel("False Positive Rate"); ax[1].set_ylabel("True Positive Rate")
        ax[1].set_title(f"ROC curve: {name}{suffix}"); ax[1].legend(loc="lower right")
        plt.tight_layout()
        plt.savefig("figs/" + re.sub(r"[^A-Za-z0-9]+", "_", name + suffix).strip("_") + ".png", dpi=150, bbox_inches="tight")
        plt.show()

def save_checkpoint():
    """Saves all results so far (protects you if Colab disconnects)."""
    with open(f"{SAVE_DIR}/checkpoint.pkl", "wb") as f:
        pickle.dump(dict(results=results, ext_results=ext_results, preds=preds, ext_preds=ext_preds, liar_in_results=liar_in_results), f)

## Step 6: ML baselines: TF-IDF + Logistic Regression and Linear SVM

In [ ]:
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), stop_words="english", sublinear_tf=True)
Xtr = tfidf.fit_transform(train_df["text"]); Xte = tfidf.transform(test_df["text"])
ytr, yte = train_df["label"].values, test_df["label"].values

lr = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
evaluate("TF-IDF + LogReg", yte, lr.predict(Xte), lr.predict_proba(Xte)[:, 1])

svm = LinearSVC().fit(Xtr, ytr)
evaluate("TF-IDF + SVM", yte, svm.predict(Xte), svm.decision_function(Xte))

### Leakage check: near-duplicates between train and test

WELFake merges several sources, so near-copies of a training article can sit in the test set and inflate accuracy. For every test article we find its most similar training article (cosine similarity of TF-IDF vectors). Test articles with similarity of **0.8 or more** are flagged; the final tables also report accuracy on the **clean** test articles only.

In [ ]:
def max_cosine_to_train(Xq, Xref, chunk=500):
    out = np.zeros(Xq.shape[0]); XrefT = Xref.T.tocsr()
    for i in range(0, Xq.shape[0], chunk):
        sims = Xq[i:i + chunk] @ XrefT
        out[i:i + chunk] = sims.max(axis=1).toarray().ravel()
    return out

NEAR_DUP = 0.8
max_sim = max_cosine_to_train(Xte, Xtr)
clean_mask = max_sim < NEAR_DUP
print(f"test articles with a near-duplicate in train (similarity >= {NEAR_DUP}): {(~clean_mask).sum()} of {len(clean_mask)} ({100*(~clean_mask).mean():.1f}%)")
print("clean test articles:", clean_mask.sum())
plt.hist(max_sim, bins=40); plt.axvline(NEAR_DUP, color="r", ls="--"); plt.xlabel("max cosine similarity to any training article"); plt.ylabel("test articles")
plt.title("Near-duplicate check"); plt.savefig("figs/near_duplicate_check.png", dpi=150, bbox_inches="tight"); plt.show()

## Step 7: Load LIAR (optional second dataset for the cross-dataset test)

LIAR has 6 labels. We map: true / mostly-true / half-true -> REAL; barely-true / false / pants-fire -> FAKE. The text is the short political `statement`. If you did not upload LIAR, this cell says so and the notebook continues with WELFake only.

In [ ]:
liar_cols = ["id", "label6", "text", "subject", "speaker", "job", "state", "party",
             "c_barely", "c_false", "c_half", "c_mostly", "c_pants", "context"]
real_set = {"true", "mostly-true", "half-true"}

def load_liar(path):
    d = pd.read_csv(path, sep="\t", header=None, names=liar_cols)
    d["label"] = d["label6"].map(lambda s: 1 if s in real_set else 0)
    return d[["text", "label"]].dropna().reset_index(drop=True)

liar_files = [p for p in glob.glob("data/**/*", recursive=True) if p.lower().endswith((".tsv", ".txt"))]
liar_test_p = [p for p in liar_files if "test" in os.path.basename(p).lower()]
liar_train_p = [p for p in liar_files if "train" in os.path.basename(p).lower()]
ext_df = None
if liar_test_p:
    ext_df = load_liar(liar_test_p[0])
    print("LIAR test file:", liar_test_p[0], " shape:", ext_df.shape)
    print(ext_df["label"].value_counts().rename({0: "FAKE", 1: "REAL"}))
    Xext = tfidf.transform(ext_df["text"]); yext = ext_df["label"].values
    evaluate("TF-IDF + SVM", yext, svm.predict(Xext), svm.decision_function(Xext), kind="ext")
    evaluate("TF-IDF + LogReg", yext, lr.predict(Xext), lr.predict_proba(Xext)[:, 1], kind="ext")
else:
    print("LIAR not found: skipping the cross-dataset test. The main results (WELFake) are unaffected.")

## Step 8: Transformer fine-tuning (one function, one cell per model)

All four models use the same data, learning rate 2e-5, weight decay 0.01 and 2 epochs, so the comparison is fair. DeBERTa uses a shorter input and a smaller batch because it needs more memory. Results are checkpointed to Drive after each model.

In [ ]:
def run_transformer(name, model_id, max_len=256, epochs=2, bs=16, fp16=True):
    tok = AutoTokenizer.from_pretrained(model_id)
    enc = lambda b: tok(b["text"], truncation=True, max_length=max_len)
    tr = Dataset.from_pandas(train_df[["text", "label"]]).map(enc, batched=True)
    te = Dataset.from_pandas(test_df[["text", "label"]]).map(enc, batched=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_id, num_labels=2)
    args = TrainingArguments(output_dir=f"out_{name}", per_device_train_batch_size=bs, per_device_eval_batch_size=32,
                             num_train_epochs=epochs, learning_rate=2e-5, weight_decay=0.01,
                             fp16=fp16 and torch.cuda.is_available(), save_strategy="no",
                             report_to="none", seed=SEED, logging_steps=100)
    trainer = Trainer(model=model, args=args, train_dataset=tr, data_collator=DataCollatorWithPadding(tok))
    trainer.train()

    out = trainer.predict(te); prob = softmax(out.predictions, axis=1)[:, 1]
    evaluate(name, test_df["label"].values, out.predictions.argmax(1), prob, kind="in")

    if ext_df is not None:
        ex = Dataset.from_pandas(ext_df[["text", "label"]]).map(enc, batched=True)
        out2 = trainer.predict(ex); prob2 = softmax(out2.predictions, axis=1)[:, 1]
        evaluate(name, ext_df["label"].values, out2.predictions.argmax(1), prob2, kind="ext")

    save_checkpoint()
    del model, trainer; gc.collect(); torch.cuda.empty_cache()

In [ ]:
run_transformer("DistilBERT", "distilbert-base-uncased")

In [ ]:
run_transformer("BERT", "bert-base-uncased")

In [ ]:
run_transformer("RoBERTa", "roberta-base")

In [ ]:
# DeBERTa: fp16 off for stability and a smaller batch (it is the most memory-hungry)
run_transformer("DeBERTa", "microsoft/deberta-base", max_len=128, bs=8, fp16=False)

## Step 9: Results tables

**In-domain** = trained and tested on WELFake. **Clean subset** = same models, but only test articles with no near-duplicate in the training set.

In [ ]:
res = pd.DataFrame(results).T.round(4).sort_values("Accuracy", ascending=False)
print("IN-DOMAIN (WELFake test set)"); display(res)

rows = {}
for n, (yt, yp, pb) in preds.items():
    m = clean_mask
    rows[n] = dict(Accuracy=accuracy_score(yt[m], yp[m]), F1=f1_score(yt[m], yp[m]), N=int(m.sum()))
clean_res = pd.DataFrame(rows).T.round(4).sort_values("Accuracy", ascending=False)
clean_res["N"] = clean_res["N"].astype(int)
print("\nCLEAN SUBSET (test articles without a near-duplicate in train)"); display(clean_res)

if ext_results:
    ext = pd.DataFrame(ext_results).T.round(4).sort_values("Accuracy", ascending=False)
    print("\nCROSS-DATASET (trained on WELFake, tested on LIAR)"); display(ext)
else:
    ext = pd.DataFrame()

## Step 10: Graphs for every model and the comparison charts

In [ ]:
def grid_plots(pred_store, title, fname):
    """One confusion matrix + one ROC curve for EVERY model, in a single figure."""
    if not pred_store: return
    names = sorted(pred_store, key=lambda n: -accuracy_score(pred_store[n][0], pred_store[n][1]))
    rows = int(np.ceil(len(names) / 2))
    fig, axs = plt.subplots(rows, 4, figsize=(19, 3.8 * rows)); axs = np.atleast_2d(axs)
    for k, name in enumerate(names):
        r, c = k // 2, (k % 2) * 2
        yt, yp, pb = pred_store[name]
        ConfusionMatrixDisplay(confusion_matrix(yt, yp), display_labels=["FAKE", "REAL"]).plot(ax=axs[r, c], cmap="Blues", colorbar=False)
        axs[r, c].set_title(f"Confusion matrix: {name}")
        fpr, tpr, _ = roc_curve(yt, pb); a = auc(fpr, tpr)
        axs[r, c + 1].plot(fpr, tpr, label=f"AUC = {a:.4f}"); axs[r, c + 1].plot([0, 1], [0, 1], "k--")
        axs[r, c + 1].set_title(f"ROC curve: {name}"); axs[r, c + 1].set_xlabel("FPR"); axs[r, c + 1].set_ylabel("TPR"); axs[r, c + 1].legend(loc="lower right")
    for k in range(len(names), rows * 2):
        axs[k // 2, (k % 2) * 2].axis("off"); axs[k // 2, (k % 2) * 2 + 1].axis("off")
    fig.suptitle(title, fontsize=15); plt.tight_layout(rect=[0, 0, 1, 0.97])
    plt.savefig(f"figs/{fname}.png", dpi=150, bbox_inches="tight"); plt.show()

grid_plots(preds, "All models: confusion matrices and ROC curves (WELFake test set)", "grid_in_domain")
grid_plots(ext_preds, "All models: confusion matrices and ROC curves (trained on WELFake, tested on LIAR)", "grid_cross_dataset_LIAR")

In [ ]:
COLORS = ["#2b6cb0", "#dd8a3a", "#2f855a", "#c0455a"]

def metric_bars(table, title, fname):
    cols = ["Accuracy", "Precision", "Recall", "F1"]
    fig, ax = plt.subplots(figsize=(10, 4.8)); x = np.arange(len(table)); w = 0.2
    for i, (c, col) in enumerate(zip(cols, COLORS)):
        bars = ax.bar(x + (i - 1.5) * w, table[c].values, w, label=c, color=col)
        for b in bars: ax.text(b.get_x() + b.get_width() / 2, b.get_height() + 0.01, f"{b.get_height():.3f}", ha="center", fontsize=6.5, rotation=90)
    ax.axhline(0.5, ls=":", color="grey", lw=0.8)
    ax.set_xticks(x); ax.set_xticklabels(table.index, rotation=20, ha="right"); ax.set_ylim(0, 1.15)
    ax.set_ylabel("Score"); ax.set_title(title); ax.legend(ncol=4, loc="upper right", frameon=False)
    ax.spines[["top", "right"]].set_visible(False); plt.tight_layout()
    plt.savefig(f"figs/{fname}.png", dpi=150, bbox_inches="tight"); plt.show()

metric_bars(res, "In-domain performance (WELFake test set)", "bars_in_domain")
if len(ext): metric_bars(ext, "Cross-dataset performance (trained on WELFake, tested on LIAR)", "bars_cross_dataset_LIAR")

In [ ]:
# Accuracy of every model: WELFake vs clean subset vs LIAR
tab = pd.DataFrame({"WELFake test": res["Accuracy"], "WELFake clean subset": clean_res["Accuracy"]})
if len(ext): tab["LIAR (cross-dataset)"] = ext["Accuracy"]
tab = tab.sort_values("WELFake test", ascending=False)
ax = tab.plot(kind="bar", figsize=(10, 4.6), color=["#2b6cb0", "#6aa3d6", "#dd8a3a"][:tab.shape[1]], ylim=(0, 1.1))
ax.axhline(0.9, ls="--", color="grey", lw=0.8); ax.axhline(0.5, ls=":", color="grey", lw=0.8)
ax.set_ylabel("Accuracy"); ax.set_title("Accuracy of every model on each test set"); plt.xticks(rotation=20, ha="right")
for cont in ax.containers: ax.bar_label(cont, fmt="%.3f", fontsize=7, rotation=90, padding=2)
ax.spines[["top", "right"]].set_visible(False); plt.tight_layout(); plt.savefig("figs/accuracy_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(1, 2 if ext_roc_store else 1, figsize=(13 if ext_roc_store else 6.5, 5)); ax = np.atleast_1d(ax)
for n, (fpr, tpr, a) in roc_store.items(): ax[0].plot(fpr, tpr, label=f"{n} (AUC={a:.3f})")
ax[0].plot([0, 1], [0, 1], "k--"); ax[0].set_title("ROC: all models (WELFake test)"); ax[0].set_xlabel("FPR"); ax[0].set_ylabel("TPR"); ax[0].legend(loc="lower right", fontsize=8)
if ext_roc_store:
    for n, (fpr, tpr, a) in ext_roc_store.items(): ax[1].plot(fpr, tpr, label=f"{n} (AUC={a:.3f})")
    ax[1].plot([0, 1], [0, 1], "k--"); ax[1].set_title("ROC: all models (LIAR)"); ax[1].set_xlabel("FPR"); ax[1].set_ylabel("TPR"); ax[1].legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.savefig("figs/roc_all_models.png", dpi=150, bbox_inches="tight"); plt.show()

## Step 11: Verify that the numbers are real and consistent

Checks that (1) accuracy printed in the table equals (TP + TN) / N computed from the confusion matrix, (2) every model was tested on the same articles, and (3) the models really make different predictions (they are not copies of each other).

In [ ]:
print(f"WELFake test set size: {len(test_df)}   |   every model tested on the same N: {sorted({len(v[0]) for v in preds.values()})}")
print(f"{'model':<18}{'N':>6}{'TN':>7}{'FP':>6}{'FN':>6}{'TP':>7}{'acc from matrix':>17}{'acc in table':>14}{'wrong':>7}")
for n, (yt, yp, pb) in preds.items():
    tn, fp, fn, tp = confusion_matrix(yt, yp).ravel()
    a = (tp + tn) / len(yt)
    assert abs(a - results[n]["Accuracy"]) < 1e-9, f"{n}: table and confusion matrix disagree!"
    print(f"{n:<18}{len(yt):>6}{tn:>7}{fp:>6}{fn:>6}{tp:>7}{a:>17.4f}{results[n]['Accuracy']:>14.4f}{fp + fn:>7}")

names = list(preds)
agree = pd.DataFrame([[np.mean(preds[a][1] == preds[b][1]) for b in names] for a in names], index=names, columns=names).round(4)
print("\nFraction of test articles on which two models give the SAME prediction (1.0 would mean identical models):")
display(agree)
print("All checks passed." if True else "")

## Step 12 (optional): LIAR when the model is trained on LIAR itself

Models trained on WELFake do badly on LIAR because the data is different. This experiment trains RoBERTa **on LIAR's own training file** and tests on LIAR's test file. It shows the realistic ceiling on LIAR. Published results for LIAR are far below 90%, so expect a score in the 60s, not 90+. Set `RUN_LIAR_INDOMAIN = False` to skip it.

In [ ]:
RUN_LIAR_INDOMAIN = True
if RUN_LIAR_INDOMAIN and ext_df is not None and liar_train_p:
    ltr = load_liar(liar_train_p[0]); print("LIAR train:", ltr.shape)
    mid, mlen = "roberta-base", 128
    tok = AutoTokenizer.from_pretrained(mid)
    enc = lambda b: tok(b["text"], truncation=True, max_length=mlen)
    tr = Dataset.from_pandas(ltr).map(enc, batched=True); te = Dataset.from_pandas(ext_df[["text", "label"]]).map(enc, batched=True)
    model = AutoModelForSequenceClassification.from_pretrained(mid, num_labels=2)
    args = TrainingArguments(output_dir="out_liar_in", per_device_train_batch_size=16, per_device_eval_batch_size=32, num_train_epochs=3,
                             learning_rate=2e-5, weight_decay=0.01, fp16=torch.cuda.is_available(), save_strategy="no", report_to="none", seed=SEED)
    trainer = Trainer(model=model, args=args, train_dataset=tr, data_collator=DataCollatorWithPadding(tok)); trainer.train()
    out = trainer.predict(te)
    evaluate("RoBERTa", ext_df["label"].values, out.predictions.argmax(1), softmax(out.predictions, axis=1)[:, 1], kind="liar_in")
    display(pd.DataFrame(liar_in_results).T.round(4))
    save_checkpoint(); del model, trainer; gc.collect(); torch.cuda.empty_cache()
else:
    print("Skipped (LIAR train/test files not found or RUN_LIAR_INDOMAIN is False).")

## Step 13: Methodology diagram

In [ ]:
from matplotlib.patches import FancyBboxPatch
steps = ["WELFake\n(title + text)", "Cleaning &\nleak removal", "Stratified\n80/20 split", "TF-IDF +\nSVM / LogReg", "Fine-tune\nDistilBERT, BERT\nRoBERTa, DeBERTa",
         "Accuracy, P, R, F1\nConfusion matrix\nROC-AUC", "Cross-dataset\ntest (LIAR)"]
fig, ax = plt.subplots(figsize=(16, 3)); ax.axis("off"); ax.set_xlim(0, len(steps) * 2); ax.set_ylim(0, 2)
for i, s in enumerate(steps):
    ax.add_patch(FancyBboxPatch((i * 2 + 0.1, 0.4), 1.6, 1.2, boxstyle="round,pad=0.05", fc="#cfe2ff", ec="#0b5ed7"))
    ax.text(i * 2 + 0.9, 1.0, s, ha="center", va="center", fontsize=9)
    if i < len(steps) - 1:
        ax.annotate("", xy=((i + 1) * 2 + 0.08, 1.0), xytext=(i * 2 + 1.74, 1.0), arrowprops=dict(arrowstyle="->"))
plt.title("Methodology"); plt.savefig("figs/methodology.png", dpi=200, bbox_inches="tight"); plt.show()

## Step 14: Save everything to Google Drive

Run this **last**, after all cells above have finished. It saves the tables (CSV), every graph (PNG) and a zip of all graphs.

In [ ]:
res.to_csv(f"{SAVE_DIR}/results_welfake.csv")
clean_res.to_csv(f"{SAVE_DIR}/results_welfake_clean_subset.csv")
if len(ext): ext.to_csv(f"{SAVE_DIR}/results_liar_cross_dataset.csv")
if liar_in_results: pd.DataFrame(liar_in_results).T.round(4).to_csv(f"{SAVE_DIR}/results_liar_trained_on_liar.csv")
save_checkpoint()
os.makedirs(f"{SAVE_DIR}/figs", exist_ok=True)
for f in glob.glob("figs/*.png"): shutil.copy(f, f"{SAVE_DIR}/figs")
shutil.make_archive(f"{SAVE_DIR}/all_figures", "zip", "figs")
print("Saved to:", SAVE_DIR); print(sorted(os.listdir(SAVE_DIR)))
try:
    from google.colab import files
    files.download(f"{SAVE_DIR}/all_figures.zip")   # also downloads the graphs to your computer
except Exception as e:
    print("Download skipped:", e)

## How to read the results

- **In-domain scores of about 95-99%** mean the models classify articles like those in WELFake very well. Transformers should beat TF-IDF.
- **Clean subset:** if accuracy barely changes, near-duplicates are not driving the result.
- **LIAR scores near 50%** mean the models do not transfer to short political claims. This is a known limitation, not a bug. Training on LIAR itself (Step 12) gives the realistic ceiling.
- Look at precision and recall per class, not only accuracy, and at the confusion matrices.